# প্রম্পট ইঞ্জিনিয়ারিংয়ের পরিচিতি
প্রম্পট ইঞ্জিনিয়ারিং হল প্রাকৃতিক ভাষা প্রক্রিয়াকরণ কাজের জন্য প্রম্পট ডিজাইন এবং অপ্টিমাইজ করার প্রক্রিয়া। এতে সঠিক প্রম্পট নির্বাচন, তাদের পরামিতি সমন্বয়, এবং তাদের কার্যকারিতা মূল্যায়ন অন্তর্ভুক্ত। প্রম্পট ইঞ্জিনিয়ারিং NLP মডেলগুলিতে উচ্চ নির্ভুলতা এবং দক্ষতা অর্জনের জন্য অত্যন্ত গুরুত্বপূর্ণ। এই অংশে, আমরা এক্সপ্লোরেশনের জন্য OpenAI মডেল ব্যবহার করে প্রম্পট ইঞ্জিনিয়ারিংয়ের বুনিয়াদি আলোচনা করব।


### ব্যায়াম ১: টোকেনাইজেশন
OpenAI থেকে একটি ওপেন-সোর্স দ্রুত টোকেনাইজার tiktoken ব্যবহার করে টোকেনাইজেশন অন্বেষণ করুন
আরও উদাহরণের জন্য দেখুন [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) ।


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### ব্যায়াম ২: মাইক্রোসফ্ট ফাউন্ড্রি মডেলস কী সেটআপ যাচাই করুন

> **নোট:** গিটহাব মডেলস ২০২৬ সালের জুলাইয়ের শেষে অবসর নিচ্ছে। এই ব্যায়ামে পরিবর্তে [মাইক্রোসফ্ট ফাউন্ড্রি মডেলস](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) ব্যবহার করা হয়েছে, যা একই ফ্রি-টু-ট্রাই মডেল ক্যাটালগ এবং আজুর এআই ইনফারেন্স এসডিকে অভিজ্ঞতা অফার করে।

নিচের কোডটি চালান যাতে আপনার মাইক্রোসফ্ট ফাউন্ড্রি মডেলস এন্ডপয়েন্ট সঠিকভাবে সেটআপ হয়েছে কিনা যাচাই করা যায়। কোডটি শুধু একটি সাধারণ বেসিক প্রম্পট চেষ্টা করে এবং কমপ্লিশন যাচাই করে। ইনপুট `oh say can you see` এর কমপ্লিশন হওয়া উচিত `by the dawn's early light..` এর মত কিছু।


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### অনুশীলন ৩: কাল্পনিক তৈরি
পরীক্ষা করুন যখন আপনি LLM-কে এমন একটি টপিক সম্পর্কে একটি প্রম্পটের জন্য সম্পূর্ণতা ফেরত দিতে বলেন যা থাকতে নাও পারে, অথবা এমন টপিক সম্পর্কে যা এটি জানে না কারণ তা তার পূর্ব-প্রশিক্ষিত ডেটাসেটের বাইরে (নতুনতর)। দেখুন প্রতিক্রিয়া কিভাবে পরিবর্তিত হয় যদি আপনি একটি ভিন্ন প্রম্পট বা একটি ভিন্ন মডেল ব্যবহার করেন।


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### ব্যায়াম ৪: নির্দেশনা ভিত্তিক 
প্রাথমিক বিষয়বস্তু সেট করতে "text" ভেরিয়েবলটি ব্যবহার করুন 
এবং সেই প্রাথমিক বিষয়বসঙ্গ সম্পর্কিত নির্দেশনা দিতে "prompt" ভেরিয়েবলটি ব্যবহার করুন।

এখানে আমরা মডেলটিকে একটি দ্বিতীয়-শ্রেণির ছাত্রের জন্য পাঠ্যের সারসংক্ষেপ করতে বলছি


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### অনুশীলন ৫: জটিল অনুরোধ 
এমন একটি অনুরোধ চেষ্টা করুন যা সিস্টেম, ব্যবহারকারী এবং সহকারী বার্তা রয়েছে 
সিস্টেম সহকারীর প্রেক্ষাপট নির্ধারণ করে
ব্যবহারকারী ও সহকারী বার্তাগুলো বহু-পর্যায়ের কথোপকথন প্রেক্ষাপট প্রদান করে

লক্ষ্য করুন কিভাবে সহকারীর ব্যক্তিত্ব সিস্টেম প্রেক্ষাপটে "বিদ্রূপাত্মক" হিসাবে সেট করা হয়েছে। 
ভিন্ন ব্যক্তিত্ব প্রেক্ষাপট ব্যবহার করে দেখুন। অথবা ভিন্ন ইনপুট/আউটপুট বার্তার সিরিজ চেষ্টা করুন


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### অনুশীলন: আপনার অন্তর্দৃষ্টি অন্বেষণ করুন
উপরের উদাহরণগুলি আপনাকে প্যাটার্ন দেয় যা আপনি নতুন প্রম্পট তৈরি করতে ব্যবহার করতে পারেন (সহজ, জটিল, নির্দেশ ইত্যাদি) - আমরা যে অন্যান্য ধারণাগুলি নিয়ে আলোচনা করেছি যেমন উদাহরণ, সংকেত এবং আরও অনেক কিছু অন্বেষণ করতে অন্যান্য অনুশীলন তৈরি করে দেখুন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
